# Deploy OHIF Viewer App

Serves the OHIF viewer, MONAI, redaction and VLM APIs;
reverse-proxies all DICOMweb protocol traffic to the gateway.

In [0]:
%pip install --upgrade databricks-sdk==0.88.0 fsspec -q
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

In [0]:
app_name = "pixels-dicomweb"
app_gateway_name = "pixels-dicomweb-gateway"
serving_endpoint_name = "pixels-monai-uc"

# Generate app.yml

In [0]:
import os

# Compute repo root from notebook path
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

_dicomweb_path = os.path.join(_repo_root, "apps", "dicom-web")

# Get gateway URL from the already-deployed gateway app
_gateway_url = w.apps.get(app_gateway_name).url

# Derive volume paths
_vol_parts = volume.split(".")
_ohif_volume_path = f"/Volumes/{_vol_parts[0]}/{_vol_parts[1]}/{_vol_parts[2]}/ohif"

# Generate app.yml from template
with open(f"{_dicomweb_path}/app-config.yml", "r") as config_input:
    with open(f"{_dicomweb_path}/app.yml", "w") as config_output:
        config_output.write(
            config_input.read()
            .replace("{PIXELS_TABLE}", table)
            .replace("{DICOMWEB_GATEWAY_URL}", _gateway_url)
            .replace("{OHIF_VOLUME_PATH}", _ohif_volume_path)
        )
print(f"Generated app.yml (gateway={_gateway_url}, OHIF_STATIC_DIR={_ohif_volume_path})")

# Create and Deploy App

In [ ]:
from databricks.sdk.service.apps import (
    AppResource,
    AppResourceSqlWarehouse,
    AppResourceSqlWarehouseSqlWarehousePermission,
    AppResourceServingEndpoint,
    AppResourceServingEndpointServingEndpointPermission,
    App,
    AppDeployment,
)

sql_resource = AppResource(
    name="sql_warehouse",
    sql_warehouse=AppResourceSqlWarehouse(
        id=sql_warehouse_id,
        permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
    ),
)

resources = [sql_resource]

if serving_endpoint_name in [ep.name for ep in w.serving_endpoints.list()]:
    resources.append(
        AppResource(
            name="serving_endpoint",
            serving_endpoint=AppResourceServingEndpoint(
                name=serving_endpoint_name,
                permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY,
            ),
        )
    )

# Viewer mints the user's x-forwarded-access-token via its app proxy and
# tunnels it to the gateway via X-Pixels-User-Token. The viewer's scopes
# determine what scopes that user token carries downstream — the gateway's
# OBO SQL/Volume calls require these.
VIEWER_USER_API_SCOPES = ["sql", "files.files"]

# Create or retrieve existing app
if app_name in [a.name for a in w.apps.list()]:
    print(f"App '{app_name}' already exists — updating resources")
    app = w.apps.update(
        app_name,
        App(
            app_name,
            resources=resources,
            user_api_scopes=VIEWER_USER_API_SCOPES,
        ),
    )
else:
    print(f"Creating DICOMweb App '{app_name}' — this may take a few minutes …")
    app = App(
        app_name,
        default_source_code_path=_dicomweb_path,
        resources=resources,
        user_api_scopes=VIEWER_USER_API_SCOPES,
    )
    app = w.apps.create_and_wait(app)
    print(f"✓ App created: {app.url}")

In [0]:
app_deploy = w.apps.deploy_and_wait(app_name, AppDeployment(source_code_path=_dicomweb_path))
print(f"✅ Viewer deployed: {app_deploy.status.message}")
print(f"   URL: {w.apps.get(app_name).url}")